# Reproducing Wirth et al. (2017) from CRCNS hc-12

Paper. Wirth S, Baraduc P, Plante A, Pinede S, Duhamel JR (2017). Gaze-informed, task-situated
representation of space in primate hippocampus during virtual navigation. PLoS Biology 15(2):
e2001045. [doi:10.1371/journal.pbio.2001045](https://doi.org/10.1371/journal.pbio.2001045).
The claims file (`claims/wirth-2017.yaml`) was written from the version of record (PMC5328243),
Results, Methods and the legends of Figs 1 to 7 and S1 to S7; S1 Appendix was not available.

Datasets. CRCNS [hc-12](https://doi.org/10.6080/K0R49NQV) (189 cells, the new-maze sessions;
every claim rests on it) and CRCNS [hc-21](https://doi.org/10.6080/K0GX48R9) (101 cells, a
familiar and a new maze per file), streamed from the public bucket `crcnsarchive`
(`hc-12/live/data.tar.gz`, `hc-21/live/data.tar.gz`). CRCNS data sets are not versioned; the
version is recorded as `unversioned`, and the MD5 sums from each data set's `checksums.md5`
for the files read are: hc-12 `data.tar.gz` 426c7d00c0358f2cc9873e0dbb331e2f (descriptor
version 0.7, 17 Jan 2017); hc-21 `data.tar.gz` dee5e8aa4a7b3c44b61b8cecfe622a0a (descriptor
version 0.52, 21 Dec 2018). Each data set has one published state (`live/`, with only
documentation under `orig/`), so the version pinned is the only one there is, and hc-12 is the
deposit the paper cites. hc-21 is read only to check whether it holds recordings that are also
in hc-12 and whether it holds the probe sessions hc-12 lacks; no claim rests on it.

| Claim | Tested here |
|---|---|
| gaze-anticipates-action, gaze-anticipates-landmarks | tested (eye-signal sign assumed, checked on the slow phases) |
| learning-trials, learning-to-90 | tested |
| probe-transfer | tested on monkey K; monkey S has no probe session in hc-12 |
| probe-faster-criterion | blocked: methods underspecified (learning-curve method of reference 60) |
| cells-recorded | tested |
| responsive-cells, multi-space-coding, sparsity-gaze-vs-state, spatial-selectivity-fraction | blocked: the point-of-gaze space cannot be built (no eye calibration); three-space results shown |
| state-space-ic-higher | tested on two of its three comparisons (the third needs gaze) |
| state-space-most-responsive | tested on the state-space count, which needs no gaze |
| direction-center-vs-periphery, direction-tuning-not-maintained | tested; reading (a) fails, reading (b) needs the responsive set, bounded |
| sparsity-state-vs-direction, sparsity-state-vs-position | tested |
| state-space-selectivity, landmark-identity-distance, landmark-identity-center | tested by bounds on the responsive set |
| path-selectivity, gaze-aligned-dynamics, latency-skew, eccentricity-modulation, foveation-selectivity, gaze-map-landmarks | blocked: need calibrated gaze |
| fields-per-cell, dimensionality-control | blocked: methods underspecified (S1 Appendix) |

In [ ]:
import resource
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import wirth

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

## One pass over hc-12

The archive is streamed once. For each cell file the helper keeps the new-maze trials, the
camera rows inside the new-maze session, the spikes and the eye signal, computes every map
measure with 999 surrogates under both surrogate readings, the landmark ANOVAs, and, once per
session, the behaviour. Seeds are fixed per cell (CRC32 of the file name).

In [ ]:
t0 = time.time()
metas, cells, beh = wirth.collect("hc-12", log=None)
M = pd.DataFrame(metas)
C_all = pd.DataFrame(cells)
print(f"hc-12 pass: {time.time() - t0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
t0 = time.time()
fp21 = pd.DataFrame(wirth.fingerprint("hc-21", log=None))
print(f"hc-21 pass: {time.time() - t0:.0f} s, {len(fp21)} files")

## Data check

In [ ]:
print("Files by monkey (ke = monkey K, sn = monkey S):")
print(M.groupby("monkey").agg(files=("file", "size"), with_new_maze=("has_newallo", "sum"),
                              sessions=("session", "nunique")).to_string())
print("\nFiles without a new-maze (newallo) field:", M.loc[~M.has_newallo, "file"].tolist())
print("\nSpike times taken from:", M.spike_src.value_counts().to_dict())
print("Top-level field sets:", M.fields.nunique(), "different layouts")
print("Camera frame interval (ms):", M.frame_dt_ms.describe()[["min", "max"]].round(3).to_dict())
print("Eye sample interval (ms):", M.eye_dt_ms.describe()[["min", "max"]].round(3).to_dict(),
      "; files without eye data:", int((~M.has_eye.fillna(False).astype(bool)).sum() - (~M.has_newallo).sum()))
print("Spike times sorted in every file:", bool(M.spikes_sorted.dropna().all()))
rng_cols = ["x_min", "x_max", "y_min", "y_max", "theta_min", "theta_max", "abscissa_max"]
print("\nRanges of the camera variables over files (min / max of each):")
print(M[rng_cols].agg(["min", "max"]).round(2).to_string())
print("\nTrials per new-maze session (per file):", M.n_trials.describe()[["min", "50%", "max"]].to_dict())
print("Session length, first trial start to last trial end (min):",
      ((M.t_end - M.t_start) / 60).describe()[["min", "50%", "max"]].round(1).to_dict())
print("\nCamera-data layout: columns of time_60 per file:", M.t60_cols.value_counts().to_dict())
print("Transition tag (column 11) over files:", M.tag_min.min(), "to", M.tag_max.max(),
      "; hundreds present per file:", M.tag_hundreds.value_counts().to_dict())
print("Next tag (column 9) over files:", M.tag_next_min.min(), "to", M.tag_next_max.max())
print("State-space radius (column 8) over files:", round(M.ss_radius_min.min(), 3), "to", round(M.ss_radius_max.max(), 3),
      "; polar angle (column 7):", round(M.ss_angle_min.min(), 3), "to", round(M.ss_angle_max.max(), 3))
print("Camera orientation range per file, sessions by (min, max) rounded to 90 deg:",
      M.dropna(subset=["theta_min"]).drop_duplicates("session").apply(
          lambda r: (int(90 * round(r.theta_min / 90)), int(90 * round(r.theta_max / 90))), axis=1).value_counts().to_dict())
odd = M[(M.abscissa_max > 1.001) | ((M.theta_max - M.theta_min) < 300)]
print("\nFiles with an abscissa above 1 or an orientation range under 300 deg:")
print(odd[["file", "abscissa_max", "abscissa_over_1", "theta_min", "theta_max", "n_trials"]].round(4).to_string())
print("Abscissa minus distance from the centre over 16 m, median absolute value per file: entry and exit paths (tags 1xy, 4xy)",
      M.abscissa_vs_r_arms.agg(["min", "max"]).round(4).to_dict(), "; passive returns (tags 5xy)",
      M.abscissa_vs_r_returns.agg(["min", "max"]).round(3).to_dict())
print("Entry-path frames with abscissa above 1, over the cells of these files:",
      C_all.set_index("name").reindex(odd.file).n_entry_frames_abscissa_over_1.to_dict())
print("Mean camera orientation on each entry path (tags 10x), sets of headings per session:",
      M.drop_duplicates("session").entry_headings.dropna().apply(lambda h: tuple(sorted(h.values()))).value_counts().to_dict())

The descriptor (hc-12, version 0.7) gives the positions as virtual X, Y. The values span x from
-16 to 12.9 and y from -15.2 to 15.2, five arm ends at radius 16 m (Methods: radius 16 m), so
positions are in metres. The camera orientation is in degrees, on 0 to 360 in some files and
-180 to 180 in others; the analysis takes it modulo 360. Column 5 (curvilinear abscissa along
the branch) is, on the entry and exit paths (tags 1xy and 4xy), the distance from the centre
divided by the 16 m arm: it runs from 0 at the centre to 1 at the arm end (median absolute
difference at most 0.0023 in every file, printed above). On the passive returns (tags 5xy) it
does not follow the distance from the centre; it runs along the return path.
Camera rows are at 60 Hz and eye samples at 200 Hz (5 ms), where the descriptor says 250 Hz.

Every file holds the 12-column camera layout of the descriptor (not the 11 columns that
`star_remapped_cell_r.m` documents), with transition tags 101 to 555 and a state-space radius
of 0.81 to 3.0, inside the -3 to 3 grid of the state-space maps. The polar angle is on -pi to pi
in some files and 0 to 2 pi in others; the maps use its cosine and sine. The mean orientation on
each entry path falls on 18, 90, 162, 234 or 306 degrees in every session, the five arms 72
degrees apart. ke20140207 spans only -126 to 90 degrees because its headings (18, 90, 234 and
306, that is -126 and -54) all fall in that range. In sn20110610 3.6% of the frames have an
abscissa up to 1.618, none of them on an entry path, so the landmark ANOVAs do not use them.

In [ ]:
B = pd.DataFrame([{k: v for k, v in b.items() if k in ("monkey", "session", "n_trials", "learningtrial", "has_eye", "eye_scale", "eye_saturated")}
                  for b in beh.values()])
lc_ok = M.lc_order_ok.dropna()
print("Learning curves: column 2 <= column 1 <= column 3 in", int(lc_ok.sum()), "of", len(lc_ok), "files")
crit_files = []
for b in beh.values():
    crit = wirth.lower_bound_criterion(b["lc"][:, 1])
    crit_files.append((b["learningtrial"], crit))
same = sum((np.isnan(c) and lt == -2) or lt == c for lt, c in crit_files)
print("Field learningtrial equals the first trial with column 2 > 1/4 (or -2 when never):", same, "of", len(crit_files), "sessions")
print("\nEye signal scale by session:", B.eye_scale.value_counts(dropna=False).to_dict())
print("Fraction of eye samples at the +-5000 / +-5 rail, by session: median",
      round(B.eye_saturated.median(), 4), "max", round(B.eye_saturated.max(), 3))
rew = pd.Series({s: tuple(np.unique(b["chosen_arm"][b["correct"]]).astype(int)) for s, b in beh.items()})
print("\nArm chosen on rewarded trials (1-5 = codes 50-54), sessions per value:", rew.value_counts().to_dict())
print("\nCamera abscissa, distance from the centre over 16 m, and fraction of frames on a passive return (tags 5xy)")
print("at event times and 1 s before and after, median over sessions:")
ev = pd.DataFrame([{k: v for k, v in b["abscissa_at_events"].items()} for b in beh.values()]).median()
clk = pd.DataFrame({"abscissa": {f"{e} {sh:+.0f}s": ev[(e, sh)] for e in ("start", "center", "reward") for sh in (-1.0, 0.0, 1.0)},
                    "r/16": {f"{e} {sh:+.0f}s": ev[(e + " r/16", sh)] for e in ("start", "center", "reward") for sh in (-1.0, 0.0, 1.0)},
                    "on return": {f"{e} {sh:+.0f}s": ev[(e + " on return", sh)] for e in ("start", "center", "reward") for sh in (-1.0, 0.0, 1.0)}})
print(clk.round(2).to_string())
tim = pd.DataFrame([{"monkey": b["monkey"], **b["timing"]} for b in beh.values()])
print("\nMedian time per session (s): reward (code 11) to trial end (code 23), and trial end to the next trial start:")
print(tim.groupby("monkey").median().round(2).to_string())


Checks against the description:
- The descriptor says the learning curve's middle column is the probability of a correct
  response, with the bounds in columns 1 and 3. In every file column 2 is the lowest and column
  3 the highest, so column 1 is the probability, column 2 the lower bound and column 3 the upper
  bound. The field `learningtrial`, which the descriptor does not describe, equals the first
  trial whose column 2 exceeds 1/4 (S3B legend's criterion), which confirms that reading.
- The descriptor says code 43 is the rewarded start arm and code 53 the rewarded chosen arm. The
  arm chosen on rewarded trials differs between sessions (printed above), as each day had a new
  reward location (Results); 53 is the rewarded arm only in some sessions.
- The eye signal is in undocumented units, on a +-5000 scale in some sessions and a +-5 scale in
  others, with a rail at the bound; no calibration to degrees is given. Eye samples are 5 ms
  apart (200 Hz), where the descriptor says 250 Hz.
- The camera rows share the trial clock. At the trial start (code 21) the camera is at the arm
  end (distance over 16 m, 1.00), and the row still belongs to the passive return (0.99 of
  them), whose abscissa has run down to 0.01; one second later the camera is on the entry path
  near the centre (0.12). One second before the arrival at the choice point (code 22) it is at
  0.72 on the entry path, at the arrival it is at the centre (0.00), and it is still there one
  second later, while the view rotates. At the reward (code 11) it is at the arm end (1.00), and
  one second earlier at 0.21 on the exit path (printed above).
- The trial ends (code 23) at the reward (code 11): the median time between them is 0 s in both
  monkeys, and the return lasts until the next trial start (median 3.6 to 3.8 s).

In [ ]:
C = C_all[C_all.n_spikes_window > 100].copy()
print("Cells with a new maze:", len(C_all), "; with more than 100 spikes in the new-maze session:", len(C),
      C.groupby("monkey").size().to_dict())
sh_cols = [c for c in C.columns if c.startswith("shift_ic|")]
resp_state = C["icp|('full', 'state')|permute"] < 0.01
rel = C.loc[resp_state, sh_cols].div(C.loc[resp_state, "shift_ic|0.0"], axis=0)
print("\nClock check for the spikes: state-space IC with the spikes shifted in time, relative to no shift,")
print("median over the", int(resp_state.sum()), "cells with significant state-space IC (and over all cells, unnormalized):")
print(pd.DataFrame({"relative (sig. cells)": rel.median().round(3).values,
                    "IC bits/spike (all cells)": C[sh_cols].median().round(3).values},
                   index=[c.split("|")[1] + " s" for c in sh_cols]).T.to_string())

The spike clock check selects cells at zero shift, so the peak there is partly a selection
effect; the fall of the information with shifts of 2 to 5 s is consistent with spikes and camera
sharing one clock, without proving it. The eye clock is checked with the turn codes in the
section on gaze anticipation.

In [ ]:
# Recordings deposited twice: hc-12 against hc-21, by spike count and first spike times.
fp12 = M[M.has_newallo][["file", "session", "n_spikes_file", "first_spikes"]].copy()
fp12["head"] = fp12.first_spikes.apply(lambda t: tuple(np.round(t[:5], 3)))
fp21m = fp21[fp21.has_maze].copy()
fp21m["head"] = fp21m.spikes_head.apply(lambda t: tuple(np.round(t[:5], 3)))
pairs = fp12.merge(fp21m, on="head", suffixes=("_hc12", "_hc21"))
print("hc-21 files:", len(fp21), "; with a new maze:", int(fp21.has_maze.sum()),
      "; sessions:", fp21.session.nunique(), fp21.drop_duplicates("session").session.str[:2].value_counts().to_dict())
print("hc-12 / hc-21 file pairs with the same first five spike times:", len(pairs))
if len(pairs):
    print(pairs[["file_hc12", "file_hc21", "n_spikes_file", "n_spikes", "n_trials"]].to_string())
print("Sessions (dates) in both data sets:", sorted(set(M.session) & set(fp21.session)))
dup12 = fp12.groupby(["head", "n_spikes_file"]).file.apply(list)
print("hc-12 files sharing spike trains with another hc-12 file:", [v for v in dup12 if len(v) > 1])

## cells-recorded

> Of 270 cells recorded in the right hippocampus (128 in monkey S, 142 in monkey K), 189 fired
> more than 100 spikes per session and were analysed.

In [ ]:
cr = pd.DataFrame({"files": M.groupby("monkey").size(),
                   "with_new_maze": M.groupby("monkey").has_newallo.sum(),
                   "over_100_spikes": C.groupby("monkey").size()})
cr.loc["total"] = cr.sum()
print(cr.to_string())
print("\nRecording locations (AP, ML, DV relative to the interaural line) present in", int(M.reclocation.notna().sum()), "files;",
      "the files give no subfield (CA3, CA1, dentate gyrus).")

**Result: partially reproduced.** hc-12 holds 189 cell files (99 of monkey K, 90 of monkey S),
the paper's analysed count; 187 have new-maze data and 186 of them fire more than 100 spikes in
the new-maze session (expected 189 +- 18.9). The 270 cells recorded (128 S, 142 K) are not all
deposited: only the 189 are, so that count cannot be checked here.

## Learning: learning-trials and learning-to-90

> Both monkeys learned the rewarded arm in about a dozen trials; monkey S performed above chance
> after 12.3 plus or minus 2 trials (SEM) and monkey K after 14.8 plus or minus 2.4.

The curves are the files' own (columns as established in the data check). Probe sessions are
defined in the probe-transfer section; reading (b) leaves them out. Sessions whose lower bound
never exceeds 1/4 have no trial to criterion and are counted apart.

In [ ]:
def probe_info(b):
    sa = b["start_arm"]
    first = set(sa[:10][np.isfinite(sa[:10])].astype(int))
    if len(first) > 2:
        return False, None, first
    intro = next((i for i in range(len(sa)) if np.isfinite(sa[i]) and int(sa[i]) not in first), None)
    return intro is not None, intro, first


rows = []
for s, b in beh.items():
    lc = b["lc"]
    crit = wirth.lower_bound_criterion(lc[:, 1])
    up = [u for u in np.nonzero(lc[:, 2] >= 0.9)[0] if np.isfinite(crit) and u + 1 >= crit]
    is_probe, intro, first = probe_info(b)
    rows.append({"session": s, "monkey": b["monkey"], "n_trials": b["n_trials"], "probe": is_probe,
                 "intro": intro, "first_arms": sorted(first), "crit": crit,
                 "t90": (up[0] + 1 - crit) if up else np.nan, "pc": b["correct"].mean(),
                 "ub_at_crit": lc[int(crit) - 1, 2] if np.isfinite(crit) else np.nan})
S = pd.DataFrame(rows)

In [ ]:
for name, sub in (("(a) all sessions", S), ("(b) sessions that are not probe sessions", S[~S.probe])):
    g = sub.groupby("monkey").crit.agg(mean="mean", sem="sem", sessions_reaching="count",
                                       sessions_never=lambda x: int(x.isna().sum()), median="median")
    print(name); print(g.round(2).to_string()); print()
print("Expected: monkey S (sn) 12.3 +- 2, monkey K (ke) 14.8 +- 2.4")

**Result: reproduced.** Under reading (a) monkey S reaches criterion after 12.26 +- 3.28 trials
(mean +- SEM, 23 sessions) and monkey K after 16.53 +- 2.69 (38 sessions), both within the
paper's 12.3 +- 2 and 14.8 +- 2.4. Without the probe sessions (b) monkey K gives 15.82 +- 3.44,
also within. Two sessions of S and three of K never reach the criterion and are left out. The
SEMs here are larger than the paper's.

In [ ]:
for name, sub in (("(a) all sessions", S), ("(b) not probe sessions", S[~S.probe])):
    reach = sub.t90.notna()
    print(f"{name}: sessions reaching both criteria {int(reach.sum())} of {len(sub)}; "
          f"fewer than 10 trials from lower bound > 1/4 to upper bound >= 0.9: {int((sub.t90 < 10).sum())}, "
          f"fraction of sessions reaching {(sub.t90 < 10).sum() / reach.sum():.3f}, of all sessions {(sub.t90 < 10).mean():.3f}")
print("Trials from criterion to upper bound 0.9 (sessions reaching):", S.t90.describe()[["min", "50%", "max"]].to_dict())
print("Expected 0.89 +- 0.089")
print("\nSessions where the upper bound is already at 0.9 or above at the criterion trial (0 trials):", int((S.t90 == 0).sum()),
      "; trials to 0.9 per session:", S.t90.value_counts().sort_index().to_dict())
print("Upper bound at the criterion trial, quartiles:", S.ub_at_crit.quantile([0.25, 0.5, 0.75]).round(3).to_dict())
print("Variant with the 0-trial sessions left out:", f"{int(((S.t90 > 0) & (S.t90 < 10)).sum())} of {int((S.t90 > 0).sum())}",
      f"= {((S.t90 > 0) & (S.t90 < 10)).sum() / (S.t90 > 0).sum():.3f}")


**Result: reproduced.** Of the 59 sessions that reach both criteria, 53 need fewer than 10
trials (0.898, expected 0.89 +- 0.089); counted over all 66 sessions the fraction is 0.803, also
inside the tolerance (0.801 to 0.979). Without the probe sessions the fractions are 0.898 and
0.815. Only one session has its upper bound at 0.9 already at the criterion trial (0 trials);
the median is 2 trials, and the upper bound at the criterion trial has a median of 0.835.
Leaving out that session gives 52 of 58 (0.897).

## probe-transfer

> In probe sessions, performance after the new western entries were introduced was higher than
> at the start of the session (monkey K 73% against 44%, monkey S 80% against 55%).

A probe session is read from the start arms: the first 10 trials use one or two entries and
later trials use others (Results; S3C legend: one entry for 36 trials, or two entries until trial
60). The introduction is the first trial from an entry not used in the first 10. The side of the
first entries is checked from the arm-end positions, relative to the rewarded arm (north by the
paper's convention). Variants: the first 5 trials from a new entry (ii); sessions whose first
entries lie clockwise of the reward, east if x is east and y north (iii), or on one side of the
rewarded arm whichever side (iii'); sessions whose introduction comes after trial 20 (iv).

In [ ]:
def arm_angles(b):
    """Angle of each start arm relative to the rewarded arm, degrees, negative = clockwise seen from
    above (x east, y north, as a right turn lowers the camera orientation): with the reward north,
    the eastern entries are at -72 (northeast) and -144 (southeast)."""
    ang = np.degrees(np.arctan2(b["pos_start"][:, 1], b["pos_start"][:, 0]))
    rew = b["pos_reward"][b["correct"]]
    ra = np.degrees(np.arctan2(np.nanmedian(rew[:, 1]), np.nanmedian(rew[:, 0])))
    out = {}
    for a in np.unique(b["start_arm"][np.isfinite(b["start_arm"])]).astype(int):
        m = b["start_arm"] == a
        out[a] = int(round(((np.nanmedian(ang[m]) - ra + 180) % 360) - 180))
    return out


P = S[S.probe].copy()
P["arm_rel_deg"] = [arm_angles(beh[s]) for s in P.session]
P["first_rel"] = [sorted(a[x] for x in f) for a, f in zip(P.arm_rel_deg, P.first_arms)]
P["eastern_first"] = P.first_rel.apply(lambda v: all(-150 <= x <= -60 for x in v))
P["one_side_first"] = P.first_rel.apply(lambda v: all(x < 0 for x in v) or all(x > 0 for x in v))
P["pc_first5"] = [beh[s]["correct"][:5].mean() for s in P.session]
P["pc_after5"] = [beh[s]["correct"][int(i):int(i) + 5].mean() for s, i in zip(P.session, P.intro)]
P["pc_new5"] = [beh[s]["correct"][[k for k in range(int(i), beh[s]["n_trials"])
                                      if np.isfinite(beh[s]["start_arm"][k]) and int(beh[s]["start_arm"][k]) not in f][:5]].mean()
                for s, i, f in zip(P.session, P.intro, P.first_arms)]
P["starts"] = ["".join(str(int(a)) if np.isfinite(a) else "." for a in beh[s]["start_arm"][:50]) for s in P.session]
print(P[["session", "monkey", "n_trials", "first_arms", "first_rel", "intro", "pc_first5", "pc_after5", "pc_new5"]].to_string())
print("\nStart arms of the first 50 trials:")
print(P[["session", "starts"]].to_string(index=False))
print("\nProbe sessions per monkey:", P.groupby("monkey").size().to_dict(), "(paper: 9 for S, 15 for K)")
print("Sessions of monkey S whose first 10 trials use at most two entries:",
      int(((S.monkey == "sn") & S.first_arms.apply(len).le(2)).sum()))
variants = (("(i) all, 5 trials from the introduction", P, "pc_after5"),
            ("(ii) all, the first 5 trials from a new entry", P, "pc_new5"),
            ("(iii) first entries eastern (clockwise of the reward), 5 trials from the introduction", P[P.eastern_first], "pc_after5"),
            ("(iii') first entries on one side of the reward arm, 5 trials from the introduction", P[P.one_side_first], "pc_after5"),
            ("(iv) introduction after trial 20, 5 trials from it", P[P.intro >= 20], "pc_after5"))
for nm, d, col in variants:
    w = stats.wilcoxon(d[col], d.pc_first5) if len(d) > 1 else None
    print(f"{nm}: {len(d)} sessions, first 5 {100 * d.pc_first5.mean():.1f}%, after {100 * d[col].mean():.1f}%,",
          f"Wilcoxon signed-rank p = {w.pvalue:.4f}" if w else "")
print("\nExpected: K 73 +- 7.3 after, 44 +- 4.4 at the start; S 80 +- 8, 55 +- 5.5")
# hc-21: sessions whose first 10 start codes use at most two entries
fp21m["first_arms"] = fp21m.codes_head.apply(lambda c: sorted(set(c)))
p21 = fp21m[fp21m.first_arms.apply(len) <= 2].drop_duplicates("session")
print("hc-21 new-maze sessions whose first 10 trials use at most two entries:", p21.session.tolist())


**Result: not reproduced.** hc-12 holds 12 probe sessions of monkey K (the paper has 15) and
none of monkey S (the paper has 9); hc-21 adds no other. For K, performance is 28.3% over the
first 5 trials and 58.3% over the 5 trials after the new entries, both below the paper's 44% and
73% by more than the tolerance, and the Wilcoxon signed-rank test gives p = 0.072. The direction
of the effect is the paper's (median gain 0.30). The first entries are not always on one side:
their angles from the rewarded arm are -72 or -144 degrees in 2 sessions, +72 or +144 in 7, and
on both sides in 3, where the paper describes the northeastern and southeastern entries. In no variant are both of K's means within tolerance: 60.0% after with the
first 5 new-entry trials (p = 0.057); 15.6% and 62.2% (p = 0.023) over the 9 sessions with the
first entries on one side; 16.0% and 64.0% (p = 0.125) over the 5 sessions introduced after
trial 20.

## gaze-anticipates-action

> During the 500 ms before each joystick action, gaze anticipated the direction of the
> subsequent movement.

Each left (code 2) or right (code 1) joystick move; mean horizontal eye signal in the 500 ms
(reading a) or 300 ms (reading b) before it, for successful trials (Method W) and, as a variant,
all trials. The eye signal is in uncalibrated units on two scales (data check), so values are
standardized within each session (minus the session median, over the session's SD) before
pooling sessions; per-session differences are also counted. Wilcoxon rank-sum, right against
left, per monkey (reading i) and pooled (reading ii). Per-session differences are
also tested across sessions (Wilcoxon signed-rank), as the turns of one session are not
independent.

In [ ]:
T = pd.DataFrame([dict(session=s, monkey=b["monkey"], **t) for s, b in beh.items() for t in b["turns"]])
print("Orientation change in the 600 ms after the code, median (deg):", T.groupby("code").dtheta_0_600ms.median().round(1).to_dict(),
      "(code 1 = right lowers the camera orientation)")


def zsess(df, col):
    g = df.groupby("session")[col]
    return (df[col] - g.transform("median")) / g.transform("std")


def rank_test(df, col, right):
    x = df[col].dropna()
    a, b = x[right.loc[x.index]], x[~right.loc[x.index]]
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan, len(a), len(b)
    return float(a.median() - b.median()), float(stats.mannwhitneyu(a, b).pvalue), len(a), len(b)

In [ ]:
res = []
for trials_used, Tsub in (("successful", T[T.correct]), ("all", T)):
    Tsub = Tsub.copy()
    for w in ("eye500", "eye300"):
        for eye in ("mean", "l", "r"):
            col = f"{w}_{eye}"
            Tsub["z"] = zsess(Tsub, col)
            for mk in ("ke", "sn", "both"):
                d = Tsub if mk == "both" else Tsub[Tsub.monkey == mk]
                diff, p, na, nb = rank_test(d, "z", d.code == 1)
                sess_diff = d.groupby("session").apply(lambda g: g[g.code == 1][col].median() - g[g.code == 2][col].median()).dropna()
                res.append({"trials": trials_used, "window": w[3:] + " ms", "eye": eye, "monkey": mk,
                            "n_right": na, "n_left": nb, "median z right-left": round(diff, 3), "p": p,
                            "sessions right<left": f"{int((sess_diff < 0).sum())}/{len(sess_diff)}",
                            "session signed-rank p": stats.wilcoxon(sess_diff).pvalue})
R1 = pd.DataFrame(res)
print(R1.to_string())
lags = [c for c in T.columns if c.startswith("lag")]
Tc = T[T.correct].copy()
prof = {}
for mk in ("ke", "sn"):
    d = Tc[Tc.monkey == mk].copy()
    for c in lags:
        d[c] = zsess(d, c)
    prof[mk] = (d[d.code == 1][lags].median() - d[d.code == 2][lags].median()).values
print("\nEye clock check: median standardized eye signal, right minus left turns, in 100 ms windows from the code:")
print(pd.DataFrame(prof, index=[f"{float(c[3:]):+.1f}" for c in lags]).T.round(2).to_string())

In [ ]:
# Sign of the eye signal from the slow phases: during a turn the scene flows across the screen
# opposite to the turn, and the slow phases of optokinetic nystagmus and pursuit follow it. The
# median of the sample-to-sample velocity over 0 to 600 ms from the code follows the slow phases.
rows = []
for mk in ("ke", "sn"):
    d = Tc[Tc.monkey == mk].copy()
    for col in ("vel_0_600ms", "vel_pre500"):
        d["zv"] = d[col] / d.groupby("session")[col].transform("std")
        diff, p, na, nb = rank_test(d, "zv", d.code == 1)
        sd = d.groupby("session").apply(lambda g: g[g.code == 1][col].median() - g[g.code == 2][col].median()).dropna()
        rows.append({"monkey": mk, "window": {"vel_0_600ms": "0 to 600 ms after the code", "vel_pre500": "500 ms before"}[col],
                     "median right": round(d[d.code == 1].zv.median(), 3), "median left": round(d[d.code == 2].zv.median(), 3),
                     "p": p, "sessions right>left": f"{int((sd > 0).sum())}/{len(sd)}", "session signed-rank p": stats.wilcoxon(sd).pvalue})
print("Median horizontal eye velocity (session SD units), right (code 1) and left (code 2) turns, successful trials:")
print(pd.DataFrame(rows).to_string())


In [ ]:
# Is the view already rotating before the code? Orientation change over the 500 ms before it
# (negative = to the right), and the position test on turns with little rotation before the code.
rows = []
for mk in ("ke", "sn"):
    d = Tc[Tc.monkey == mk]
    for code, nm in ((1, "right"), (2, "left")):
        x = d[d.code == code].dtheta_pre500.dropna()
        rows.append({"monkey": mk, "turn": nm, "n": len(x), "median deg": round(x.median(), 2),
                     "IQR": f"{x.quantile(0.25):.2f} to {x.quantile(0.75):.2f}",
                     "fraction rotating the way of the turn": round(float(((x < 0) if code == 1 else (x > 0)).mean()), 3)})
print("Camera orientation change over the 500 ms before the turn code, successful trials:")
print(pd.DataFrame(rows).to_string())
rows = []
for mk in ("ke", "sn"):
    d = Tc[Tc.monkey == mk].copy()
    d["z"] = zsess(d, "eye500_mean")
    d["zv"] = d["vel_pre500"] / d.groupby("session")["vel_pre500"].transform("std")
    for lim in (np.inf, 1.0, 0.5, 0.25):
        s = d[d.dtheta_pre500.abs() < lim]
        diff, p, na, nb = rank_test(s, "z", s.code == 1)
        vdiff, vp, _, _ = rank_test(s, "zv", s.code == 1)
        rows.append({"monkey": mk, "|rotation before| below (deg)": lim, "n_right": na, "n_left": nb,
                     "position: median z right-left": round(diff, 3), "p": p,
                     "velocity before: median right-left": round(vdiff, 3), "p velocity": vp})
print("\nPosition (500 ms before, both eyes) and eye velocity before the code, by the rotation before it:")
print(pd.DataFrame(rows).to_string())


Sign of the eye signal. The files do not say which sign of the horizontal signal is rightward.
It is taken from the slow phases. During a turn the scene flows across the screen opposite to
the turn, and the slow phases of optokinetic nystagmus and smooth pursuit follow the scene, so
the eye drifts to the left during a right turn. The median eye velocity from 0 to 600 ms after
the code is positive during right turns and negative during left turns, in 40 of 40 sessions of
K and 20 of 20 of S (session signed-rank p = 1.8e-12 and 1.9e-6). The slow phases are therefore
leftward when the signal increases, and a negative value is rightward. This uses the eye
velocity only, not the eye position that the claim tests. The mean position during the turn
moves the other way, toward the turn (profile above), as the mean eye position does during
optokinetic stimulation. The sign is still an assumption, now supported by this check.

Is the view already rotating in the 500 ms before the code? In both monkeys it is, slowly: the
camera orientation changes by a median of -0.49 and 0.54 degrees (K) and -0.75 and 0.69
degrees (S) over those 500 ms, in the direction of the coming turn in 95% to 99% of turns, as if
the code came after the turn had begun. In K the eye velocity before the code also differs
between right and left turns (0.250 against -0.189 session SD, 38 of 40 sessions, p = 2.3e-9),
in the direction of the slow phases; in S it does not (9 of 20 sessions, p = 0.40). The
position effect remains in the turns with the least rotation before the code: with less than
0.25 degrees, K -0.27 session SD (p = 3.3e-5, 355 and 328 turns) and S -0.54 (p = 0.0059, 32 and
106 turns); with less than 1 degree, K -0.37 and S -0.695. In K the velocity difference also
remains with less than 0.25 degrees (0.248, p = 2.6e-10), so it is not explained by the
rotation of the view, and part of K's position effect may continue an eye movement under way
before the window. In K's low-rotation turns the position effect is a third to a half of
that over all turns (-0.77). The paper's S2B legend shows second turns of a series of two turns, another control
for this; it was not run here.

**Result: reproduced, resting on the sign of the eye signal.** Before right turns the
horizontal eye signal lies on the negative (rightward) side of that before left turns in each
monkey, in both windows, for each eye and both eyes, with rank-sum p below 1e-100 (500 ms, both
eyes, successful trials: K median difference -0.77 session SD, p = 4.0e-148; S -1.00,
p = 4.3e-146). Across sessions the difference is negative in 39 of 40 sessions of K and 20 of 20
of S (signed-rank p = 1.3e-11 and 1.9e-6), so the result does not rest on pooling the turns of
a session.

## gaze-anticipates-landmarks

> When relocated to a new entry, the monkeys gazed proactively at the location where the
> landmarks would appear.

Passive returns (tags 5xy between the end of a trial and the next start); a return turns right
when the camera orientation falls over it (as for code 1). Mean horizontal eye signal over the
return, standardized within session; rank-sum right against left. Returns after all trials,
and after successful trials as a variant (Method W).

In [ ]:
Rt = pd.DataFrame([dict(session=s, monkey=b["monkey"], **r) for s, b in beh.items() for r in b["returns"]])
res = []
for trials_used, Rsub in (("all", Rt), ("after successful", Rt[Rt.correct])):
    Rsub = Rsub.copy()
    for eye in ("mean", "l", "r"):
        Rsub["z"] = zsess(Rsub, f"eye_{eye}")
        for mk in ("ke", "sn", "both"):
            d = Rsub if mk == "both" else Rsub[Rsub.monkey == mk]
            diff, p, na, nb = rank_test(d, "z", d.dtheta < 0)
            sd = d.groupby("session").apply(lambda g: g[g.dtheta < 0][f"eye_{eye}"].median() - g[g.dtheta >= 0][f"eye_{eye}"].median()).dropna()
            res.append({"returns": trials_used, "eye": eye, "monkey": mk, "n_right": na, "n_left": nb,
                        "median z right-left": round(diff, 3), "p": p, "sessions right<left": f"{int((sd < 0).sum())}/{len(sd)}",
                        "session signed-rank p": stats.wilcoxon(sd).pvalue})
print("Median absolute orientation change over a return (deg):", round(Rt.dtheta.abs().median(), 1))
print(pd.DataFrame(res).to_string())
vrows = []
for mk in ("ke", "sn"):
    d = Rt[Rt.monkey == mk].copy()
    d["zv"] = d.vel / d.groupby("session").vel.transform("std")
    diff, p, na, nb = rank_test(d, "zv", d.dtheta < 0)
    sd = d.groupby("session").apply(lambda g: g[g.dtheta < 0].vel.median() - g[g.dtheta >= 0].vel.median()).dropna()
    vrows.append({"monkey": mk, "median right": round(d[d.dtheta < 0].zv.median(), 3), "median left": round(d[d.dtheta >= 0].zv.median(), 3),
                  "p": p, "sessions right>left": f"{int((sd > 0).sum())}/{len(sd)}"})
print("\nMedian horizontal eye velocity over the return (session SD units), right- and left-turning returns, all returns:")
print(pd.DataFrame(vrows).to_string())


**Result: reproduced, resting on the sign of the eye signal.** During passive returns that
turn right the eye signal lies on the negative (rightward) side of that during returns that turn
left, in each monkey (all returns, both eyes: K -0.56 session SD, p = 6.2e-58, 39 of 41 sessions,
signed-rank across sessions p = 1.3e-9; S -0.56, p = 6.4e-30, 20 of 21 sessions, p = 1.9e-6),
and after successful trials only (p = 4.7e-34 and 1.5e-10). The sign comes from the slow phases
during turns at the centre (above); the eye velocity over the returns gives the same sign
(right-turning returns higher in 40 of 41 sessions of K and 21 of 21 of S). The test measures
where the eyes are during the return, as the claims file's criterion does; whether this is
anticipation or the eye position that goes with any rotation of the view cannot be told apart
here.

## Information content in the coding spaces

The point-of-gaze space needs the gaze direction in degrees, which the files cannot give (the
eye signal is uncalibrated, data check). Position, direction and state space are computed for
the cells with more than 100 spikes in the new-maze session. Readings: trial window "full"
(start to next start), "toreward" (start to reward, no return) or "toreward_ret" (start to
reward plus the return that follows); surrogates "permute" (chunk order permuted) or "shift"
(each 5 s chunk rotated within itself by its own random offset) (Method W). Significance is the
bilateral surrogate p < 0.01; the upper tail alone is shown as a variant. The responsive set of
the paper needs all four spaces; the set responsive in the three spaces here is contained in
it, and all cells contain it, so counts over these two sets bound the count over the paper's set.

The surrogates are first checked on synthetic spike trains that code nothing: a homogeneous
Poisson train on a smooth random trajectory over 200 bins. A valid surrogate gives about 1% of
such trains p < 0.01. The first version of the "shift" reading, which moved each chunk by an
offset over the whole session so that chunks overlapped, is run alongside for comparison.

In [ ]:
def shift_overlapping(s, rng, L=300):
    """The first version of the "shift" reading: each chunk moved by an independent offset
    over the whole session, circularly, so that chunks overlap (replaced)."""
    T = len(s)
    src = np.arange(T)
    ch = src // L
    tgt = (src + rng.integers(0, T, size=ch[-1] + 1)[ch]) % T
    return np.bincount(tgt, weights=s, minlength=T)


rng = np.random.default_rng(2017)
T = 60 * 600                                             # 10 min of 60 Hz frames
walk = np.cumsum(rng.normal(0, 1, T))
walk = np.convolve(walk, np.ones(120) / 120, "same")     # a smooth trajectory
b = np.clip(((walk - walk.min()) / np.ptp(walk) * 200).astype(int), 0, 199)
occ = np.bincount(b, minlength=200) * wirth.DT
valid = occ > 0
rows = []
for name in ("permute", "shift", "shift, overlapping chunks (replaced)"):
    ps, ratio = [], []
    for k in range(60):
        s = rng.poisson(2.0 * wirth.DT, T).astype(float)
        a = wirth.skaggs_ic(np.bincount(b, weights=s, minlength=200)[valid], occ[valid])
        if name.startswith("shift, overlapping"):
            surr = np.vstack([shift_overlapping(s, rng) for _ in range(999)])
        else:
            surr = np.vstack(list(wirth.surrogate_counts(s, rng, name)))
        ic_s = wirth.skaggs_ic(np.stack([np.bincount(b, weights=x, minlength=200)[valid] for x in surr]), occ[valid])
        ps.append(wirth.tail_p(a, ic_s)[0])
        ratio.append(ic_s.mean() / a)
    ps = np.array(ps)
    rows.append({"surrogates": name, "trains": len(ps), "mean surrogate IC / actual IC": round(float(np.mean(ratio)), 3),
                 "fraction p < 0.01": float(np.mean(ps < 0.01)), "fraction p < 0.05": float(np.mean(ps < 0.05))})
print("Synthetic Poisson trains (2 Hz, no coding), 999 surrogates each:")
print(pd.DataFrame(rows).to_string())


In [ ]:
READINGS = [(r, sr) for r in wirth.WINDOWS for sr in ("permute", "shift")]
SPACES = ("position", "direction", "state")


def sig(space, r, sr, alpha=0.01, tail="two"):
    """IC significant: bilateral surrogate p (tail "two") or upper tail alone ("up")."""
    return C[f"{'icp' if tail == 'two' else 'icup'}|('{r}', '{space}')|{sr}"] < alpha


rows = []
for r, sr in READINGS:
    sg = {k: sig(k, r, sr) for k in SPACES}
    n_sig = sum(v.astype(int) for v in sg.values())
    rows.append({"trials": r, "surrogates": sr, **{k: int(v.sum()) for k, v in sg.items()},
                 "any of three": int((n_sig > 0).sum()), "two or more": int((n_sig >= 2).sum()),
                 "cells": len(C), "state, upper tail": int(sig("state", r, sr, tail="up").sum()),
                 "any of three, upper tail": int((sum(sig(k, r, sr, tail="up").astype(int) for k in SPACES) > 0).sum()),
                 "state, low side (lower tail < 0.005)": int((C[f"iclo|('{r}', 'state')|{sr}"] < 0.005).sum())})
IC3 = pd.DataFrame(rows)
print(IC3.to_string())
print("\nMedian number of valid bins per map:", C[[c for c in C.columns if c.startswith("nbins|")]].median().to_dict())
print("Analysed frames per cell, median, by trial window:", C[[f"frames|{r}" for r in wirth.WINDOWS]].median().to_dict())


The corrected "shift" surrogates give 3.3% of the synthetic trains p < 0.01 and 5% p < 0.05
(permuted chunks 0% and 5%), with surrogate IC within 2% of the actual IC on average; the first
version, with overlapping chunks, gave all of them p < 0.01 and surrogate IC 2.1 times the actual.
At most 3 cells (permuted chunks) and none (shifted chunks) have state-space IC significantly low,
so the bilateral test adds almost nothing from the low side. The "toreward_ret" window is
almost the "full" one (38,212 against 38,224 analysed frames per cell, median), as the trial
ends at the reward; its bilateral counts differ by at most one cell from those of "full".

## responsive-cells (blocked) and state-space-most-responsive

> 111 of the 189 cells (59%) had significant information content in at least one of the four
> coding spaces.

> The proportions of cells responsive to each space were unequal; the state space accounted for
> the most cells (84), and more cells were responsive to state space than to point of gaze.

In [ ]:
print("Cells with significant state-space IC (p < 0.01), per reading, and per monkey:")
for r, sr in READINGS:
    s_ = sig("state", r, sr)
    print(f"  {r}/{sr}: {int(s_.sum())} of {len(C)}", s_.groupby(C.monkey).sum().to_dict())
print("Expected 84 +- 8.4 (75.6 to 92.4)")
print("\nThree-space responsive cells per reading (a lower bound of the paper's 111 +- 11.1):", IC3["any of three"].tolist())
print("Upper tail alone (p < 0.01), state space:", [int(sig("state", r, sr, tail="up").sum()) for r, sr in READINGS])

**Result: not reproduced.** The state-space count is 57 of 186 cells under full trials with
permuted chunks, and 47, 42, 37, 57 and 47 under the other five readings, all below the expected
75.6 to 92.4. With the upper tail alone the counts are 41 to 58. The count needs no gaze, so the
missing gaze space does not change this; the chi-squared tests across the four spaces could not
be run. State space is the space with the most significant cells of the three computed here
under every reading (57 against 46 and 40 under the main reading).

## direction-center-vs-periphery

> For direction-selective cells, both information per spike and sparsity of directional firing
> were higher in the maze center than on the peripheral return paths.

Directional maps from centre time (tags 2xy, 3xy) and return paths (tags 5xy) of successful
trials with their returns (reading "full"), at one common angular resolution per session.
Normalized IC and sparsity (minus the surrogate mean); Wilcoxon signed-rank across cells.

In [ ]:
dir_rows = []
for sr in ("permute", "shift"):
    for cellset, mask in (("(a) direction IC p < 0.01", sig("direction", "full", sr)),
                          ("(b) three-space responsive", sum(sig(k, "full", sr).astype(int) for k in SPACES) > 0),
                          ("all cells (reference)", pd.Series(True, index=C.index))):
        d = C[mask]
        for meas in ("ic", "sp"):
            a, b_ = d[f"{meas}n|center|{sr}"], d[f"{meas}n|return|{sr}"]
            ok = a.notna() & b_.notna()
            w = stats.wilcoxon(a[ok], b_[ok])
            dir_rows.append({"surrogates": sr, "cells": cellset, "measure": {"ic": "IC", "sp": "sparsity"}[meas],
                             "n": int(ok.sum()), "median centre": round(a[ok].median(), 3),
                             "median return": round(b_[ok].median(), 3),
                             "median diff": round((a[ok] - b_[ok]).median(), 3),
                             "centre higher": int((a[ok] > b_[ok]).sum()), "p": w.pvalue})
print(pd.DataFrame(dir_rows).to_string())
print("\nValid direction bins: centre median", C["nbins|center"].median(), ", return median", C["nbins|return"].median())

**Result: blocked (metadata insufficient).** For the 40 direction-selective cells (reading a,
permuted chunks) normalized sparsity is higher in the centre (median difference 0.020,
p = 0.031) but IC is not (0.012, p = 0.43); with shifted chunks, 28 cells, p = 0.016 and 0.095.
Under reading (a) neither the reproduced level nor the partial level is met. Reading (b) needs
the paper's responsive set, which needs the gaze space. Over the cells responsive in the three
spaces computed here, which it contains, IC and sparsity are both higher in the centre with
p = 0.045 and 0.0074 (59 cells, permuted chunks), the criterion's partial level; with shifted
chunks (51 cells) IC is not (p = 0.14, sparsity p = 0.013). Over all 180 cells with both maps
neither IC nor sparsity is higher in the centre (p = 0.31 and 0.29, permuted chunks). The
signed-rank result changes with the cell set, and the result over a subset puts no bound on the
one over a larger set, so the paper's set may give any level. No set reaches p < 0.001. As
reading (a) fails and reading (b) cannot be formed, the claim cannot be decided. The verifier
agreed with this outcome in round 3.

## direction-tuning-not-maintained

> Direction tuning was not maintained across positions; the correlation between directional
> tuning in the center and on return paths was significantly low in 20 of 56 cells (36%), more
> than chance, and significantly high in only 3, consistent with chance.

Uncentered correlation over the bins valid in both maps (Methods, Directional correlations);
low or high when the actual value is in the lower or upper 2.5% of the surrogates; binomial
chance rate 0.025 (claims file). Cells with at least 10 common valid bins (a standard choice;
2 shown as a variant).

In [ ]:
rows = []
for sr in ("permute", "shift"):
    for cellset, mask in (("(a) direction IC p < 0.01", sig("direction", "full", sr)),
                          ("(b) three-space responsive", sum(sig(k, "full", sr).astype(int) for k in SPACES) > 0),
                          ("all cells", pd.Series(True, index=C.index))):
        for min_common in (10, 2):
            d = C[mask & (C.n_common >= min_common) & C[f"corr_lo|{sr}"].notna()]
            lo = int((d[f"corr_lo|{sr}"] <= 0.025).sum())
            hi = int((d[f"corr_up|{sr}"] <= 0.025).sum())
            rows.append({"surrogates": sr, "cells": cellset, "min common bins": min_common, "n": len(d),
                         "low": lo, "high": hi, "p binom low": wirth.binom_upper(lo, len(d), 0.025),
                         "p binom high": wirth.binom_upper(hi, len(d), 0.025),
                         "median corr": round(d["corr"].median(), 3) if len(d) else np.nan})
print(pd.DataFrame(rows).to_string())
print("Expected: tested 56 +- 5.6, low 20 +- 2 with binomial p < 0.001, high 2 to 4")
# Reading (b) over the paper's responsive set: that set contains the three-space responsive cells
# and lies within all cells. Can a set between the two meet the criterion? Added cells raise n by
# one each and the low or the high count by at most one.
DT_ = pd.DataFrame(rows)
print("\nReading (b) over a set between the three-space responsive cells and all cells (10 common bins):")
for sr in ("permute", "shift"):
    g = DT_[(DT_.surrogates == sr) & (DT_["min common bins"] == 10)].set_index("cells")
    s_, a_ = g.loc["(b) three-space responsive"], g.loc["all cells"]
    ways = []
    for n in range(max(s_.n, 51), min(61, a_.n) + 1):
        extra = n - s_.n
        for lo in range(max(s_.low, 18), min(22, s_.low + extra, a_.low) + 1):
            for hi in range(max(s_.high, 2), min(4, s_.high + extra - (lo - s_.low), a_.high) + 1):
                if wirth.binom_upper(lo, n, 0.025) < 0.001:
                    ways.append((n, lo, hi))
    print(f"  {sr}: three-space n {s_.n}, low {s_.low}, high {s_.high}; all cells n {a_.n}, low {a_.low}, high {a_.high};",
          f"combinations meeting the criterion: {len(ways)}" + (f", for example n {ways[0][0]}, low {ways[0][1]}, high {ways[0][2]}" if ways else ""))

**Result: blocked (metadata insufficient).** Among the 40 direction-selective cells (permuted
chunks), 15 have a significantly low centre-return correlation (38%, binomial p = 2.1e-14 at
chance 0.025) and 4 a significantly high one (p = 0.017). The proportion of low cells is close
to the paper's 36% and the high count is inside 2 to 4, but 40 cells are tested (expected
56 +- 5.6) and the low count of 15 is outside 20 +- 2; with shifted chunks 28 cells are tested,
2 high and 11 low. Reading (a) fails on the number of cells. Reading (b) needs the paper's
responsive set, which needs the gaze space. Over the three-space responsive cells, 59 are tested
(inside the tolerance), 4 are high and 16 low; with shifted chunks 51 are tested, 3 high and 14
low. Over these sets the low count misses 18 to 22, although more cells than chance have a low
correlation under all of them (binomial p below 1e-10). The minimum of common bins (10 or 2)
changes nothing. The paper's responsive set contains the three-space responsive cells and lies
within all cells (181 tested, 25 low and 8 high with permuted chunks; 18 low and 7 high with
shifted chunks). A set between the two can meet the criterion under both readings of the
surrogates: with permuted chunks in one combination (n 61, low 18, high 4), with shifted chunks
in 13 (for example n 55, low 18, high 3). A set that meets the criterion and one that fails it
both exist, so the claim cannot be decided. The verifier agreed with this outcome in round 3.

## sparsity-state-vs-direction and sparsity-state-vs-position

> Sparsity was significantly higher in state space than in direction space.

> Sparsity was not significantly higher in state space than in position space overall, but the
> 15 cells whose difference was individually significant were sparser in state space.

Cells with significant IC in both spaces compared; per cell normalized state-space sparsity
minus the other space's; Wilcoxon signed-rank. The per-cell difference is significant when its
surrogate p (the difference recomputed on each surrogate) is below 0.05 (reading a) or 0.01 (b).

In [ ]:
def sparsity_compare(other):
    rows = []
    for r, sr in READINGS:
        both = sig("state", r, sr) & sig(other, r, sr)
        d = C[both]
        diff = d[f"spn|('{r}', 'state')|{sr}"] - d[f"spn|('{r}', '{other}')|{sr}"]
        w = stats.wilcoxon(diff) if len(diff) > 1 else None
        row = {"trials": r, "surrogates": sr, "cells in both": len(d), "median diff": round(diff.median(), 4),
               "state sparser": int((diff > 0).sum()), "p": w.pvalue if w else np.nan}
        for lvl, nm in ((0.05, "a"), (0.01, "b")):
            ind = d[f"spdiffp|{r}|{other}|{sr}"] < lvl
            sub = diff[ind]
            ws = stats.wilcoxon(sub) if len(sub) > 1 else None
            row.update({f"indiv sig ({nm})": int(ind.sum()), f"their median ({nm})": round(sub.median(), 4) if len(sub) else np.nan,
                        f"their p ({nm})": ws.pvalue if ws else np.nan})
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
print(sparsity_compare("direction").to_string())

**Result: reproduced under the full-trial window, with either surrogate.** For the 40 cells
significant in both state and direction space (full trials, permuted chunks), normalized
sparsity is higher in state space (median difference 0.016, 28 of 40 cells, Wilcoxon
p = 0.0022); with shifted chunks, 27 cells, 0.020, p = 0.0059. With the start-to-reward window it
is not significant (p = 0.92 and 0.98), and the start-to-reward window with the returns gives
the full-trial numbers. The outcome rests on keeping the returns in the maps.

In [ ]:
print(sparsity_compare("position").to_string())
print("Expected: overall not significant (p >= 0.05); individually significant 15 +- 1.5, sparser in state space with p < 0.05")

**Result: not reproduced.** State space is significantly sparser than position space overall
under every reading (Wilcoxon p = 0.00013 to 0.00043, median difference 0.014 to 0.028), where
the paper finds no significant difference. The individually significant cells number 7 to 11
(level 0.05) or 3 to 5 (level 0.01), against 15 +- 1.5. No reading meets either of these two
parts. The cells that differ individually are sparser in state space (p = 0.00098 to 0.016 at
level 0.05), the third part.

## state-space-ic-higher

> Spike information content was systematically higher in state space than in position,
> direction or point-of-gaze space, for cells responsive in both spaces compared.

Cells with significant IC in state space and in the other space; per cell normalized state-space
IC minus the other space's; Wilcoxon signed-rank. The comparison with the point-of-gaze space
cannot be made (no gaze calibration); the other two are shown under every reading.

In [ ]:
rows = []
for r, sr in READINGS:
    for other in ("position", "direction"):
        both = sig("state", r, sr) & sig(other, r, sr)
        d = C[both]
        diff = d[f"icn|('{r}', 'state')|{sr}"] - d[f"icn|('{r}', '{other}')|{sr}"]
        w = stats.wilcoxon(diff) if len(diff) > 1 else None
        rows.append({"trials": r, "surrogates": sr, "other space": other, "cells in both": len(d),
                     "median diff": round(diff.median(), 4), "state higher": int((diff > 0).sum()),
                     "p": w.pvalue if w else np.nan})
ICH = pd.DataFrame(rows)
print(ICH.to_string())
print("Criterion: positive with p < 0.01; partially reproduced for two of the three comparisons")


**Result: partially reproduced.** Normalized IC is higher in state space than in position
space (full trials, permuted chunks: 44 cells, median difference 0.014, 35 higher, p = 1.1e-4)
and than in direction space (40 cells, 0.021, 35 higher, p = 2.5e-7), and both comparisons give
p < 0.01 under all six readings (largest p 0.0022). The third comparison, with the point-of-gaze
space, cannot be made without a calibration of the eye signal, so two of the three hold, the
criterion's partial level.

## state-space-selectivity

> In the maze center, at the same position and direction, 12 of 111 responsive cells (11%)
> showed significantly high state-space selectivity, discriminating the current rotational move,
> more than chance; only 4 showed significantly low selectivity, as expected by chance.

SSI per Methods, State-space selectivity; high or low when in the upper or lower 5% of the
surrogates (binomial chance 0.05, claims file). Counts over the three-space responsive cells
(lower bound) and over all cells (upper bound). The three moves are not named in the paper;
three typings are run: "arc" (36 degree alignment, 72 degree rotation right, 72 degree rotation
left), "align_dir" (36 degree alignment right, left, 72 degree rotation) and "tag" (each
transition tag of the state-space graph as its own move).

In [ ]:
rows = []
for ty in wirth.TYPINGS:
    for r, sr in READINGS:
        resp3 = sum(sig(k, r, sr).astype(int) for k in SPACES) > 0
        for cellset, mask in (("three-space responsive", resp3), ("all cells", pd.Series(True, index=C.index))):
            d = C[mask & C[f"ssi_up|{ty}|{sr}"].notna()]
            hi = int((d[f"ssi_up|{ty}|{sr}"] <= 0.05).sum())
            lo = int((d[f"ssi_lo|{ty}|{sr}"] <= 0.05).sum())
            rows.append({"moves": ty, "trials (responsive set)": r, "surrogates": sr, "cells": cellset, "n": len(d), "high": hi, "low": lo,
                         "p binom high": wirth.binom_upper(hi, len(d), 0.05), "p binom low": wirth.binom_upper(lo, len(d), 0.05)})
SSI = pd.DataFrame(rows)
print(SSI.to_string())
for ty in wirth.TYPINGS:
    print(f"Moves {ty}: cells without an SSI {int(C[f'ssi|{ty}'].isna().sum())}, bins entering the SSI, median {C[f'ssi_bins|{ty}'].median()}")
print("Expected: high 12 +- 1.2 (11 to 13) with binomial p < 0.05; low 3 to 5, not significant")
bounds = SSI.pivot_table(index=["moves", "trials (responsive set)", "surrogates"], columns="cells", values=["high", "low"])
bounds[("high can be 11 to 13", "")] = (bounds[("high", "three-space responsive")] <= 13) & (bounds[("high", "all cells")] >= 11)
bounds[("low can be 3 to 5", "")] = (bounds[("low", "three-space responsive")] <= 5) & (bounds[("low", "all cells")] >= 3)
print("\nCan the counts over the paper's responsive set (between the two rows) meet the criterion?")
print(bounds.to_string())


**Result: not reproduced under permuted chunks, an assumption; open under shifted chunks.**
The paper's responsive set contains the three-space responsive cells and is contained in all
cells, so its counts lie between the two printed rows. With permuted chunks the low count over
the three-space responsive cells is already 6 or 7 under every move typing and trial window,
above the 3 to 5 the criterion allows, so no responsive set can meet the criterion. With shifted
chunks the bounds include the targets in most combinations (for example arc moves, full trials:
high 13 to 21, low 5 to 8), so the count over the paper's set cannot be decided without the
gaze space. The outcome rests on the permuted reading of the surrogates, which the Results'
"randomly shuffling periods of spiking activity in time" supports, and does not depend on the
move typing.

## landmark-identity-distance and landmark-identity-center

> A two-way ANOVA of firing on the entry paths by landmark viewed and relative distance gave a
> main effect of landmark identity in 30 of 111 responsive cells and of relative distance in 64;
> 29 cells coded a combination (14 with a significant interaction, 15 with both main effects),
> more than chance.

> The main effect of landmark identity was confirmed by a one-way ANOVA on firing to the four
> landmark views at the center, significant in 12 of 111 responsive cells, more than chance.

Which laps enter is read two ways. Method W takes successful trials only; the Methods' ANOVA
section says "firing rates were collected for each of the animal's laps on each of the four
entry paths", and Fig 5 reports F(9,756) for two cells recorded together. In hc-12 only one
session gives 756 residual degrees of freedom: sn20110624, with 193 trials, when every lap is
used (193 x 4 - 16 = 756; printed below). Both readings are run. Counts over the
three-space responsive cells and over all cells bound the count over the paper's responsive
set, as above. The ANOVA uses type II sums of squares; type III, the
default of MATLAB's `anovan`, is shown alongside.

In [ ]:
def lm_counts(d, laps):
    a = 0.05
    k2, k1 = ("lm2", "lm1") if laps == "successful" else ("lm2all", "lm1")
    lmk, rd, it = d[f"{k2}_p_lm"] < a, d[f"{k2}_p_rd"] < a, d[f"{k2}_p_int"] < a
    ca, cb = ("lm1a_p", "lm1b_p") if laps == "successful" else ("lm1aall_p", "lm1ball_p")
    t3 = {f"{nm}, type III": int((d[f"{k2}_{c}3"] < a).sum()) for nm, c in (("identity", "p_lm"), ("distance", "p_rd"), ("interaction", "p_int"))}
    return {"laps": laps, "n": len(d), "identity": int(lmk.sum()), "distance": int(rd.sum()), "interaction": int(it.sum()), **t3,
            "both, no interaction (a)": int((lmk & rd & ~it).sum()), "both (b)": int((lmk & rd).sum()),
            "centre one-way (a)": int((d[ca] < a).sum()), "centre one-way (b)": int((d[cb] < a).sum())}


rows = []
for laps in ("successful", "all"):
    for r, sr in READINGS:
        resp3 = sum(sig(k, r, sr).astype(int) for k in SPACES) > 0
        rows.append({"responsive set": f"three-space {r}/{sr}", **lm_counts(C[resp3], laps)})
    rows.append({"responsive set": "all cells", **lm_counts(C, laps)})
LM = pd.DataFrame(rows)
print("Entry paths per cell:", C.lm2_n_lm.value_counts().to_dict(), "; laps per cell, median: successful", C.n_laps.median(),
      ", all", C.n_laps_all.median())
print("Residual df of the two-way ANOVA, median: successful laps", C.lm2_df_res.median(), ", all laps", C.lm2all_df_res.median())
ex = C[C.session == "sn20110624"][["name", "n_trials", "lm2_df_res", "lm2all_df_res", "lm2all_F_int", "lm2all_p_int"]]
print("\nThe session with 193 trials, whose residual df with all laps is 193 x 4 - 16 = 756, as in Fig 5:")
print(ex.round(4).to_string())

In [ ]:
print(LM.drop(columns=["centre one-way (a)", "centre one-way (b)"]).to_string())
print("Expected: identity 30 +- 3, distance 64 +- 6.4, interaction 14 +- 1.4, both 15 +- 1.5")
fx = C[["name", "session", "lm2all_F_int", "lm2all_p_int", "lm2all_df_int", "lm2all_df_res"]].copy()
fx["dist"] = np.minimum(abs(fx.lm2all_F_int - 4.4076), abs(fx.lm2all_F_int - 2.1243))
print("\nCells whose interaction F (all laps) comes nearest the Fig 5 values (4.41 and 2.12):")
print(fx.nsmallest(4, "dist").round(4).to_string())

**Result: not reproduced.** With successful laps, at most 24 cells (all 186 cells) show a main
effect of relative distance, and at most 31 with all laps (27 and 33 with type III sums of
squares), against 64 +- 6.4; this bound holds for any responsive set. The identity count lies
between 15 and 32 (successful laps) or 21 and 47 (all laps), which allows 30, and the
interaction count between 5 and 14 or 8 and 17, but the distance count fails under every
reading, and with it the partial level. All laps give the paper's residual df of 756 in the one
session of 193 trials; none of its three cells has the Fig 5 F values (1.83, 3.19, 1.56 against
4.41 and 2.12).

In [ ]:
print(LM[["responsive set", "laps", "n", "centre one-way (a)", "centre one-way (b)"]].to_string())
print("Expected 12 +- 1.2 (11 to 13), binomial p < 0.05 at chance 0.05")
print("Groups in the one-way ANOVAs:", C.lm1a_groups.value_counts().to_dict(), C.lm1b_groups.value_counts().to_dict())

**Result: blocked (metadata insufficient).** Over the three-space responsive cells and over all
cells the one-way count lies between 7 and 16 (reading a) or 10 and 23 (reading b) with
successful laps, and between 7 and 18 or 10 and 26 with all laps, so it may or may not fall in
11 to 13, and the binomial test depends on the size of the responsive set. Without the
point-of-gaze space the paper's responsive set cannot be formed, and the claim cannot be
decided.

## Claims blocked by the missing gaze calibration: three-space results

These numbers do not decide an outcome; they show how far the claims can be followed without
the point-of-gaze space. spatial-selectivity-fraction under reading (a) needs no gaze: the
fraction of cells with significant position IC is 0.129 to 0.247 under the six readings, outside
0.5 +- 0.05. Readings (b) and (c) need the responsive set: (b) is at least 0.253 to 0.317 and
(c) lies between 0.129 and 0.780, both of which allow 0.5. As the criterion is met when one
reading is within tolerance, the claim stays blocked.

In [ ]:
rows = []
for r, sr in READINGS:
    resp3 = sum(sig(k, r, sr).astype(int) for k in SPACES) > 0
    pos = sig("position", r, sr)
    rows.append({"trials": r, "surrogates": sr, "responsive in three spaces": int(resp3.sum()),
                 "two or more spaces": int((sum(sig(k, r, sr).astype(int) for k in SPACES) >= 2).sum()),
                 "position IC p < 0.01": int(pos.sum()), "(a) position cells among all": round(pos.mean(), 3),
                 "(b) responsive among all, at least": round(resp3.mean(), 3),
                 "(c) position among responsive, from": round(pos.sum() / len(C), 3),
                 "(c) to": round(pos.sum() / max(resp3.sum(), 1), 3)})
print(f"{len(C)} cells. spatial-selectivity-fraction expects 0.5 +- 0.05 under one of (a), (b), (c):")
print(pd.DataFrame(rows).to_string())


## Summary

| Claim | Outcome |
|---|---|
| gaze-anticipates-action | reproduced (sign of the eye signal assumed, supported by the slow phases) |
| gaze-anticipates-landmarks | reproduced (same sign) |
| learning-trials | reproduced |
| learning-to-90 | reproduced |
| probe-transfer | not reproduced (monkey K only; no probe session of S in hc-12) |
| probe-faster-criterion | blocked: methods underspecified |
| cells-recorded | partially reproduced (only the 189 analysed cells are deposited) |
| responsive-cells | blocked: metadata insufficient (no gaze calibration) |
| state-space-most-responsive | not reproduced |
| multi-space-coding | blocked: metadata insufficient |
| direction-center-vs-periphery | blocked: metadata insufficient (reading a fails; reading b needs the responsive set) |
| direction-tuning-not-maintained | blocked: metadata insufficient (reading a fails on the cell count; the paper's set could meet the criterion) |
| state-space-ic-higher | partially reproduced (two of three comparisons; the third needs gaze) |
| sparsity-state-vs-direction | reproduced (full-trial window) |
| sparsity-state-vs-position | not reproduced |
| sparsity-gaze-vs-state | blocked: metadata insufficient |
| state-space-selectivity | not reproduced (permuted chunks; open under shifted chunks) |
| landmark-identity-distance | not reproduced |
| landmark-identity-center | blocked: metadata insufficient |
| path-selectivity, gaze-aligned-dynamics, latency-skew, eccentricity-modulation, foveation-selectivity, gaze-map-landmarks | blocked: metadata insufficient |
| fields-per-cell, dimensionality-control | blocked: methods underspecified |
| spatial-selectivity-fraction | blocked: metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The point-of-gaze space is missing, as the eye
signal has no calibration, so the responsive set of the paper cannot be formed; counts over it
are bounded by counts over the three-space responsive cells and over all cells. The map
resolution steps, the surrogate shifting (two readings), the trial windows (three readings), the
move types of the state-space selectivity (three typings), the per-session standardization of
the eye signal and its sign are choices the paper does not fix. The learning-curve method of the
paper's reference 60 is not known, so curves were not recomputed.

What was found about the data. The learning curves' columns are probability, lower and upper
bound, not the order the descriptor gives; eye samples are 200 Hz, not 250 Hz; the eye signal
is on two different scales with no units, sign or calibration; the rewarded arm is not always
code 53; two hc-12 files have no new-maze data; 87 of the 101 hc-21 files hold the same spike
trains as hc-12 files; hc-12 holds 12 of monkey K's 15 probe sessions and none of monkey S's 9,
and in 3 of the 12 the first entries lie on both sides of the rewarded arm.